# Análise de Série Temporal — Preço do Caroço de Algodão (MT)
### Projeto P2 · ME607 Séries Temporais


In [ ]:
!pip install -q pymc arch statsmodels 2>/dev/null

In [ ]:
import pandas as pd, numpy as np
import matplotlib, warnings, os
matplotlib.use('Agg'); warnings.filterwarnings('ignore')
import matplotlib.pyplot as plt, matplotlib.gridspec as gridspec
from matplotlib.ticker import FuncFormatter
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.stats.diagnostic import acorr_ljungbox, het_arch
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from scipy import stats
import pymc as pm, pytensor.tensor as pt
plt.rcParams.update({'figure.facecolor':'white','axes.facecolor':'white','axes.grid':True,
  'grid.alpha':0.25,'axes.spines.top':False,'axes.spines.right':False})
AZUL='#2C6E9B'; VERM='#C0392B'; VERDE='#27AE60'; AMAR='#E67E22'
fmt=FuncFormatter(lambda x,_:f'R${x:,.0f}')
MESES=['Jan','Fev','Mar','Abr','Mai','Jun','Jul','Ago','Set','Out','Nov','Dez']
SEED = 236312
np.random.seed(SEED)

## Carregar o dado único e construir a série mensal (média das safras)

**Origem dos dados:** o preço médio mensal de comercialização do caroço de algodão é
publicado pelo IMEA (Instituto Mato-grossense de Economia Agropecuária). Para obtê-lo,
é preciso acessar o site do IMEA, solicitar o indicador, receber um link por e-mail e
baixar o arquivo. Para facilitar a reprodução deste trabalho, o banco foi espelhado em um
repositório público no GitHub e é lido diretamente da URL abaixo.

In [ ]:
URL = 'https://raw.githubusercontent.com/ThalesCrivillari/serie-temporal-caroco-algodao/refs/heads/main/caroco_p2.csv'
raw = pd.read_csv(URL)

raw['data'] = pd.to_datetime(raw['data'])
raw['am']   = raw['data'].dt.to_period('M')

df = (raw.groupby('am')['preco_rs_ton'].mean().reset_index()
      .rename(columns={'preco_rs_ton':'preco'}))
df['data'] = df['am'].dt.to_timestamp()
df['ano']  = df['data'].dt.year; df['mes_num'] = df['data'].dt.month
ts = df['preco'].values; datas = df['data']; n = len(ts); ML = min(15, n//2-2)
print(f"N = {n} meses ({df['am'].iloc[0]} a {df['am'].iloc[-1]})")
print(f"Média R${ts.mean():.0f}  DP R${ts.std():.0f}  Mín R${ts.min():.0f}  Máx R${ts.max():.0f}")

sob = raw.groupby('am').filter(lambda g: len(g)==2)
difs = [abs(g.sort_values('safra')['preco_rs_ton'].values[0]
            - g.sort_values('safra')['preco_rs_ton'].values[1]) for _,g in sob.groupby('am')]
print(f"Meses com 2 safras: {sob['am'].nunique()} | diferença média entre safras: R${np.mean(difs):.0f}/ton")
df.head()

## Série e decomposição clássica (médias móveis)

In [ ]:
fig,ax=plt.subplots(figsize=(10,3.4))
ax.plot(datas,ts,color=AZUL,lw=1.8); ax.fill_between(datas,ts,alpha=0.1,color=AZUL)
ax.yaxis.set_major_formatter(fmt); ax.set_ylabel('R$/ton')
ax.set_title('Preço médio mensal do caroço de algodão — MT (IMEA)',fontweight='bold')
plt.savefig('f1_serie.png', dpi=150, bbox_inches='tight'); plt.show()

from statsmodels.tsa.seasonal import seasonal_decompose
dec=seasonal_decompose(ts,model='additive',period=12,extrapolate_trend='freq')
stl=dec
rr=dec.resid[~np.isnan(dec.resid)]
ss=max(0,1-np.var(rr)/np.var((dec.seasonal+dec.resid)[~np.isnan(dec.resid)]))
tr=max(0,1-np.var(rr)/np.var((dec.trend+dec.resid)[~np.isnan(dec.resid)]))
fig,axes=plt.subplots(3,1,figsize=(10,5.5),sharex=True)
for ax,(lab,c,cor) in zip(axes,[('Tendência',dec.trend,AMAR),('Sazonal',dec.seasonal,VERDE),('Resíduo',dec.resid,VERM)]):
    (ax.bar(datas,c,color=cor,alpha=0.7,width=20) if lab=='Resíduo' else ax.plot(datas,c,color=cor,lw=1.6)); ax.set_ylabel(lab)
axes[0].set_title(f'Decomposição clássica (médias móveis) (força tend.={tr:.2f}, saz.={ss:.2f})',fontweight='bold')
plt.savefig('f2_stl.png', dpi=150, bbox_inches='tight'); plt.show()
print(f"Força tendência={tr:.2f}  sazonalidade={ss:.2f}")

## Estacionariedade (ADF + KPSS) e ACF/PACF

In [ ]:
def adf(x,l): r=adfuller(x,autolag='AIC'); print(f"ADF  ({l}): stat={r[0]:.3f} p={r[1]:.4f}"); return r[1]
def kp(x,l):
    s,p,_,_=kpss(x,regression='c',nlags='auto'); print(f"KPSS ({l}): stat={s:.3f} p={p:.4f}"); return p
ts_d=np.diff(ts)
print("== Nível =="); adf(ts,'nível'); kp(ts,'nível')
print("\n== 1ª diferença =="); adf(ts_d,'1a dif'); kp(ts_d,'1a dif')

fig,axes=plt.subplots(2,2,figsize=(11,6))
plot_acf(ts,lags=ML,ax=axes[0,0],color=AZUL,title='ACF — nível')
plot_pacf(ts,lags=ML,ax=axes[0,1],color=AZUL,title='PACF — nível',method='ywm')
plot_acf(ts_d,lags=ML,ax=axes[1,0],color=VERDE,title='ACF — 1ª diferença')
plot_pacf(ts_d,lags=ML,ax=axes[1,1],color=VERDE,title='PACF — 1ª diferença',method='ywm')
plt.tight_layout(); plt.savefig('f3_acfpacf.png', dpi=150, bbox_inches='tight'); plt.show()

## SARIMA — seleção por AICc e diagnóstico

In [ ]:
specs=[(1,1,0,0,0,0),(0,1,1,0,0,0),(1,1,1,0,0,0),(2,1,0,0,0,0),(2,1,1,0,0,0),
       (1,1,0,1,0,0),(0,1,1,0,0,1),(1,1,1,1,0,0)]
res=[]
for p,d,q,P,D,Q in specs:
    try:
        mm=SARIMAX(ts,order=(p,d,q),seasonal_order=(P,D,Q,12),trend='n',
                   enforce_stationarity=False,enforce_invertibility=False).fit(disp=False)
        res.append({'Modelo':f'SARIMA({p},{d},{q})({P},{D},{Q})[12]','AIC':round(mm.aic,1),
                    'BIC':round(mm.bic,1),'AICc':round(mm.aicc,1),'_o':mm})
    except: pass
tab=pd.DataFrame(res).sort_values('AICc').reset_index(drop=True)
display(tab.drop(columns='_o'))
nome=tab.iloc[0]['Modelo']; fit=next(r['_o'] for r in res if r['Modelo']==nome)
print("Melhor:",nome)

fitted=np.array(fit.fittedvalues); resid=np.array(fit.resid); resid=resid[~np.isnan(resid)]
fig=plt.figure(figsize=(11,6.5)); gs=gridspec.GridSpec(2,3,figure=fig,hspace=0.4,wspace=0.32)
a1=fig.add_subplot(gs[0,:]); a2=fig.add_subplot(gs[1,0]); a3=fig.add_subplot(gs[1,1]); a4=fig.add_subplot(gs[1,2])
a1.plot(datas,ts,color=AZUL,lw=1.8,label='Observado'); a1.plot(datas,fitted,color=VERM,lw=1.4,ls='--',label='Ajustado')
a1.yaxis.set_major_formatter(fmt); a1.legend(); a1.set_title(f'{nome} — ajuste',fontweight='bold')
(o,r2),(sl,ic,rr)=stats.probplot(resid,dist='norm'); a2.scatter(o,r2,s=15,color=AZUL); a2.plot(o,sl*np.array(o)+ic,color=VERM); a2.set_title(f'QQ (r={rr:.3f})')
plot_acf(resid,lags=ML,ax=a3,color=VERDE,title='ACF resíduos')
a4.hist(resid[~np.isnan(resid)],bins=9,color=AZUL,alpha=0.7,density=True,edgecolor='white'); a4.set_title('Histograma')
plt.savefig('f4_diag.png', dpi=150, bbox_inches='tight'); plt.show()

lb=acorr_ljungbox(resid,lags=[12],return_df=True); sw=stats.shapiro(resid); ap=het_arch(resid,nlags=4)[1]
mae=np.mean(np.abs(ts-fitted)); rmse=np.sqrt(np.mean((ts-fitted)**2)); mape=np.mean(np.abs((ts-fitted)/ts))*100
print(f"Ljung-Box(12) p={lb['lb_pvalue'].iloc[-1]:.3f} | Shapiro p={sw.pvalue:.4f} | ARCH p={ap:.3f}")
print(f"MAE=R${mae:.0f} RMSE=R${rmse:.0f} MAPE={mape:.1f}%")

## Exercício de previsão: 30 treino + 10 teste
Treina o SARIMA com as 30 primeiras observações e prevê as 10 seguintes, exibindo a tabela com observado, previsto e resíduo de previsão.

In [ ]:
NTR=30
trn3,te3 = ts[:NTR], ts[NTR:NTR+10]
datas_te3 = datas[NTR:NTR+10]
f3=SARIMAX(trn3,order=(1,1,0),seasonal_order=(1,0,0,12),trend='n',
           enforce_stationarity=False,enforce_invertibility=False).fit(disp=False)
fo=f3.get_forecast(steps=10); fc3=np.array(fo.predicted_mean); ci3=np.array(fo.conf_int(alpha=0.05))

tab=pd.DataFrame({
    'Mês':[d.strftime('%b/%Y') for d in datas_te3],
    'Observado':te3.round(0),
    'Previsto':fc3.round(0),
    'Resíduo':(te3-fc3).round(0),
    'Erro %':((te3-fc3)/te3*100).round(1),
})
display(tab)
mae3=np.mean(np.abs(te3-fc3)); rmse3=np.sqrt(np.mean((te3-fc3)**2)); mape3=np.mean(np.abs((te3-fc3)/te3))*100
dentro=int(np.sum((te3>=ci3[:,0])&(te3<=ci3[:,1])))
print(f"MAE=R${mae3:.0f}  RMSE=R${rmse3:.0f}  MAPE={mape3:.1f}%  | dentro do IC95%: {dentro}/10")

fig,ax=plt.subplots(figsize=(10,4))
ax.plot(datas[:NTR],trn3,color=AZUL,lw=1.7,label='Treino (30)')
ax.plot(datas_te3,te3,'o-',color='black',lw=1.8,ms=5,label='Observado (10)')
ax.plot(datas_te3,fc3,'s--',color=VERM,lw=1.6,ms=4,label=f'Previsto (MAE={mae3:.0f})')
ax.fill_between(datas_te3,ci3[:,0],ci3[:,1],alpha=0.15,color=VERM,label='IC 95%')
ax.axvline(datas.iloc[NTR-1],color='gray',ls=':',lw=1); ax.yaxis.set_major_formatter(fmt)
ax.legend(fontsize=8,ncol=2); ax.set_title('Previsão fora da amostra: 30 treino, 10 previsão',fontweight='bold')
plt.savefig('f7_prev3010.png', dpi=150, bbox_inches='tight'); plt.show()

## Análise de intervenção (Morettin & Toloi, cap. 12)
A série sofre uma queda abrupta de patamar em meados de 2025. Aqui identificamos a data por AIC (sem chutar), comparamos degrau vs pulso, e ajustamos o SARIMA com a intervenção. Nota: o pico de mai/2025 é só o topo da tendência; a intervenção é a QUEDA de ago/2025.

In [ ]:
print("Maiores variações mensais (1a diferença):")
d1=np.diff(ts)
for i in sorted(np.argsort(np.abs(d1))[::-1][:5]):
    print(f"  {datas.iloc[i+1].strftime('%b/%Y')}: R${d1[i]:+.0f}")

print("\nSeleção da data do degrau por AIC:")
res=[]
for k in range(20,n-2):
    step=np.zeros(n); step[k:]=1.0
    try:
        mm=SARIMAX(ts,exog=step,order=(1,1,0),seasonal_order=(1,0,0,12),trend='n',
                   enforce_stationarity=False,enforce_invertibility=False).fit(disp=False)
        res.append((k,datas.iloc[k].strftime('%b/%Y'),mm.aic,mm.params[0]))
    except: pass
res.sort(key=lambda x:x[2])
for k,dt,aic,coef in res[:4]:
    print(f"  {dt}: AIC={aic:.1f}  coef={coef:+.0f}")
kbest=res[0][0]
print(f"-> Data ótima: {res[0][1]} (AIC={res[0][2]:.1f}) vs SARIMA puro 305.4")

print("\nDegrau vs Pulso na data ótima:")
for tipo,vec in [('Degrau',np.r_[np.zeros(kbest),np.ones(n-kbest)]),('Pulso',np.eye(n)[kbest])]:
    mm=SARIMAX(ts,exog=vec,order=(1,1,0),seasonal_order=(1,0,0,12),trend='n',
               enforce_stationarity=False,enforce_invertibility=False).fit(disp=False)
    print(f"  {tipo}: AIC={mm.aic:.1f}")

step=np.zeros(n); step[kbest:]=1.0
m_int=SARIMAX(ts,exog=step,order=(1,1,0),seasonal_order=(1,0,0,12),trend='n',
              enforce_stationarity=False,enforce_invertibility=False).fit(disp=False)
fit_int=np.array(m_int.fittedvalues); fit_puro=np.array(fit.fittedvalues)
mae_int=np.mean(np.abs(ts-fit_int))
print(f"\nSARIMA + intervenção: AIC={m_int.aic:.1f}  efeito degrau=R${m_int.params[0]:.0f}/ton  MAE={mae_int:.0f}")

fig,ax=plt.subplots(figsize=(10,4))
ax.plot(datas,ts,color='black',lw=2,label='Observado',zorder=3)
ax.plot(datas,fit_puro,color=VERM,lw=1.5,ls='--',label=f'SARIMA puro (MAE={np.mean(np.abs(ts-fit_puro)):.0f})',alpha=0.8)
ax.plot(datas,fit_int,color=VERDE,lw=1.6,label=f'SARIMA + intervenção (MAE={mae_int:.0f})')
ax.axvline(datas.iloc[kbest],color='#7D3C98',ls=':',lw=2,label=f'Intervenção ({datas.iloc[kbest].strftime("%b/%Y")})')
ax.yaxis.set_major_formatter(fmt); ax.legend(fontsize=8); ax.set_ylabel('R$/ton')
ax.set_title('Modelo com análise de intervenção (degrau)',fontweight='bold')
plt.savefig('f9_intervencao.png', dpi=150, bbox_inches='tight'); plt.show()

from scipy import stats
X1,X2=ts[:kbest],ts[kbest:]
sp=np.sqrt(((len(X1)-1)*X1.var(ddof=1)+(len(X2)-1)*X2.var(ddof=1))/(len(X1)+len(X2)-2))
tstat=(X1.mean()-X2.mean())/(sp*np.sqrt(1/len(X1)+1/len(X2)))
pval=2*(1-stats.t.cdf(abs(tstat),len(X1)+len(X2)-2))
print(f"\nTeste t (sec 12.4): X1=R${X1.mean():.0f} X2=R${X2.mean():.0f} | t={tstat:.2f} p={pval:.3f}")

print("\nForma da resposta (abrupta vs gradual):")
print(f"  Abrupta (degrau):  AIC={m_int.aic:.1f}")
for delta in [0.3,0.5,0.7]:
    grad=np.zeros(n); val=0
    for t in range(kbest,n): val=delta*val+(1-delta); grad[t]=val
    mg=SARIMAX(ts,exog=grad,order=(1,1,0),seasonal_order=(1,0,0,12),trend='n',
               enforce_stationarity=False,enforce_invertibility=False).fit(disp=False)
    print(f"  Gradual (delta={delta}): AIC={mg.aic:.1f}")

## Diagnóstico do modelo com intervenção
Verifica que o modelo final (com o degrau) continua válido. Destaque: a intervenção corrige a não-normalidade dos resíduos, pois a queda de ago/2025 deixa de ser um resíduo extremo.

In [ ]:
from statsmodels.stats.diagnostic import het_arch
from scipy import stats as _st

def _diag(resid):
    r=resid[~np.isnan(resid)][5:]
    lb=acorr_ljungbox(r,lags=[12],return_df=True)['lb_pvalue'].iloc[-1]
    arch=het_arch(r,nlags=6)[1]
    sh=_st.shapiro(r)[1]; jb=_st.jarque_bera(r)[1]
    return r,lb,arch,sh,jb

_m0=SARIMAX(ts,order=(1,1,0),seasonal_order=(1,0,0,12),trend='n',
            enforce_stationarity=False,enforce_invertibility=False).fit(disp=False)
r0,lb0,ar0,sh0,jb0=_diag(_m0.resid)
r1,lb1,ar1,sh1,jb1=_diag(m_int.resid)

print(f"{'Teste':<22}{'SARIMA puro':>13}{'+ Interv.':>12}")
print(f"{'Ljung-Box(12)':<22}{lb0:>13.3f}{lb1:>12.3f}")
print(f"{'ARCH-LM(6)':<22}{ar0:>13.3f}{ar1:>12.3f}")
print(f"{'Shapiro-Wilk':<22}{sh0:>13.4f}{sh1:>12.4f}")
print(f"{'Jarque-Bera':<22}{jb0:>13.4f}{jb1:>12.4f}")
print(f"\nNormalidade: puro {'REJEITA' if sh0<0.05 else 'OK'} (p={sh0:.3f}) -> interv. {'REJEITA' if sh1<0.05 else 'OK'} (p={sh1:.3f})")

_R0=_m0.resid.copy(); _R0[:5]=np.nan
_R1=m_int.resid.copy(); _R1[:5]=np.nan
_a0=het_arch(_R0[~np.isnan(_R0)],nlags=6)[1]; _a1=het_arch(_R1[~np.isnan(_R1)],nlags=6)[1]
fig,axes=plt.subplots(2,2,figsize=(11,7))
for row,(r,R,cor,tit,pv,ap) in enumerate([(r0,_R0,AZUL,'SARIMA puro',sh0,_a0),(r1,_R1,VERDE,'com intervenção',sh1,_a1)]):
    ax=axes[row,0]; _st.probplot(r,dist='norm',plot=ax)
    ax.get_lines()[0].set_markerfacecolor(cor); ax.get_lines()[0].set_markeredgecolor(cor); ax.get_lines()[0].set_markersize(5)
    ax.get_lines()[1].set_color(VERM)
    ax.set_title(f'QQ — {tit} (Shapiro p={pv:.3f})',fontweight='bold',fontsize=10)
    ax=axes[row,1]; _sd=np.nanstd(R)
    ax.plot(datas,R,'o-',color=cor,ms=3.5,lw=0.9,alpha=0.8)
    ax.axhline(0,color='black',lw=0.9); ax.axhline(2*_sd,color=VERM,ls='--',lw=1.1); ax.axhline(-2*_sd,color=VERM,ls='--',lw=1.1)
    ax.fill_between(datas,-2*_sd,2*_sd,alpha=0.06,color=VERM)
    ax.set_title(f'Resíduos no tempo — {tit} (ARCH-LM p={ap:.3f})',fontweight='bold',fontsize=9)
    ax.set_ylabel('Resíduo (R$)'); ax.tick_params(axis='x',labelsize=7,rotation=45)
plt.tight_layout(); plt.savefig('f10_diag_intervencao.png',dpi=150,bbox_inches='tight'); plt.show()

## Modelo híbrido SARIMA→BTVC (PyMC/MCMC)
A previsão do SARIMA entra como regressor com peso β(t) variável no tempo, aprendido via MCMC. O modelo bayesiano decide quanta confiança dar ao SARIMA em cada instante.

In [ ]:
NTR=30; H=10
tr,te=ts[:NTR],ts[NTR:NTR+10]; datas_te=datas[NTR:NTR+10]
fit_s=SARIMAX(tr,order=(1,1,0),seasonal_order=(1,0,0,12),trend='n',
    enforce_stationarity=False,enforce_invertibility=False).fit(disp=False)
sar_in=np.array(fit_s.fittedvalues); sar_oos=np.array(fit_s.get_forecast(steps=H).predicted_mean)
m,s=tr.mean(),tr.std(); yn=(tr-m)/s; xn=(sar_in-m)/s
K=2; tt=np.arange(NTR,dtype=float)
F=np.column_stack([fn for k in range(1,K+1) for fn in [np.sin(2*np.pi*k*tt/12),np.cos(2*np.pi*k*tt/12)]])
with pm.Model() as hib:
    sl=pm.HalfNormal('sl',0.2); sb=pm.HalfNormal('sb',0.1); sg=pm.HalfNormal('sg',0.1)
    so=pm.HalfNormal('so',0.3); nu=pm.Gamma('nu',2,0.2)
    l0=pm.Normal('l0',float(yn[0]),1); li=pm.Normal('li',0,sl,shape=NTR-1)
    lev=pm.Deterministic('lev',pt.concatenate([[l0],l0+pt.cumsum(li)]))
    b0=pm.Normal('b0',1,0.5); bi=pm.Normal('bi',0,sb,shape=NTR-1)
    beta=pm.Deterministic('beta',pt.concatenate([[b0],b0+pt.cumsum(bi)]))
    g0=pm.Normal('g0',0,0.5,shape=2*K); gi=pm.Normal('gi',0,sg,shape=(NTR-1,2*K))
    gw=pm.Deterministic('gw',pt.concatenate([g0[None,:],g0[None,:]+pt.cumsum(gi,axis=0)],axis=0))
    mu=lev+beta*xn+pt.sum(gw*pt.as_tensor_variable(F),axis=1)
    pm.StudentT('obs',nu=nu,mu=mu,sigma=so,observed=yn)
    idata=pm.sample(800,tune=800,chains=2,cores=1,target_accept=0.92,progressbar=True,random_seed=SEED)
ll=idata.posterior['lev'].values.reshape(-1,NTR)[:,-1]
bl=idata.posterior['beta'].values.reshape(-1,NTR)[:,-1]
gl=idata.posterior['gw'].values.reshape(-1,NTR,2*K)[:,-1,:]
tf=np.arange(NTR,NTR+H,dtype=float)
Ff=np.column_stack([fn for k in range(1,K+1) for fn in [np.sin(2*np.pi*k*tf/12),np.cos(2*np.pi*k*tf/12)]])
ps=ll[:,None]+bl[:,None]*((sar_oos-m)/s)[None,:]+(gl@Ff.T)
pred=ps.mean(0)*s+m; plo=np.percentile(ps,2.5,0)*s+m; phi=np.percentile(ps,97.5,0)*s+m
tab_h=pd.DataFrame({'Mês':[d.strftime('%b/%Y') for d in datas_te],
    'Observado':te.round(0),'Previsto':pred.round(0),
    'Resíduo':(te-pred).round(0),'Erro %':((te-pred)/te*100).round(1)})
display(tab_h)
mae_h=np.mean(np.abs(te-pred)); mape_h=np.mean(np.abs((te-pred)/te))*100
print(f"Híbrido OOS: MAE=R${mae_h:.0f}  MAPE={mape_h:.1f}%  (SARIMA no mesmo teste: MAE=R$269, MAPE=29.9%)")
fig,ax=plt.subplots(figsize=(10,4))
ax.plot(datas[:NTR],tr,color=AZUL,lw=1.7,label='Treino (30)')
ax.plot(datas_te,te,'o-',color='black',lw=1.8,ms=5,label='Observado (10)')
ax.plot(datas_te,pred,'^--',color=VERDE,lw=1.6,ms=4,label=f'Híbrido (MAE={mae_h:.0f})')
ax.fill_between(datas_te,plo,phi,alpha=0.15,color=VERDE,label='IC 95%')
ax.axvline(datas.iloc[NTR-1],color='gray',ls=':',lw=1); ax.yaxis.set_major_formatter(fmt)
ax.legend(fontsize=8,ncol=2); ax.set_title('Previsão fora da amostra do híbrido (30 treino, 10 previsão)',fontweight='bold')
plt.savefig('f8_hibrido.png', dpi=150, bbox_inches='tight'); plt.show()

## Diferença de preço entre safras (interpretação da quebra)
A queda de patamar de ago/2025 coincide com o fim da sobreposição de safras. Aqui mostramos, de forma descritiva, o preço de cada safra e a diferença entre elas nos meses sobrepostos (sem ajustar modelo).

In [ ]:
import matplotlib.dates as mdates
cores_safra={'22/23':'#2C6E9B','23/24':'#27AE60','24/25':'#E67E22'}

difs=[]
for am,g in raw.groupby('am'):
    if len(g)==2:
        p=g['preco_rs_ton'].values
        difs.append((am.to_timestamp(), abs(p[1]-p[0])))
dd=[d for d,_ in difs]; dv=[v for _,v in difs]
print(f"Diferença média entre safras: R${np.mean(dv):.0f}/ton | pico R${max(dv):.0f}/ton (mai/2025)")

fig,axes=plt.subplots(1,2,figsize=(13,4.5),gridspec_kw={'width_ratios':[1.55,1]})
ax=axes[0]
for s,g in raw.groupby('safra'):
    gd=g.copy(); gd['data']=gd['am'].dt.to_timestamp()
    ax.scatter(gd['data'],gd['preco_rs_ton'],s=18,color=cores_safra.get(s,'gray'),label=f'Safra {s}',alpha=0.85,zorder=3)
ax.plot(datas,ts,color=VERM,lw=2,label='Média mensal (série usada)',zorder=2)
ax.yaxis.set_major_formatter(fmt); ax.set_ylabel('R$/ton'); ax.legend(fontsize=8,loc='upper left')
ax.set_title('(a) Preço por safra e média mensal',fontweight='bold',fontsize=11)
ax=axes[1]
ax.bar(dd,dv,width=20,color='#7D3C98',alpha=0.75)
ax.yaxis.set_major_formatter(fmt); ax.set_ylabel('Diferença entre safras (R$/ton)')
ax.axhline(np.mean(dv),color=VERM,ls='--',lw=1.3,label=f'Média R${np.mean(dv):.0f}'); ax.legend(fontsize=8)
ax.set_title('(b) Diferença entre safras sobrepostas',fontweight='bold',fontsize=11)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b/%y'))
plt.setp(ax.get_xticklabels(),rotation=45,ha='right',fontsize=7)
plt.tight_layout(); plt.savefig('f6_hier.png',dpi=150,bbox_inches='tight'); plt.show()

## Validação out-of-sample (comparação justa)

In [ ]:
H=6; trn,te=ts[:-H],ts[-H:]
ft=SARIMAX(trn,order=(1,1,0),seasonal_order=(1,0,0,12),trend='n',
           enforce_stationarity=False,enforce_invertibility=False).fit(disp=False)
fc=np.array(ft.get_forecast(steps=H).predicted_mean); mae_oos=np.mean(np.abs(te-fc))

tt=np.arange(len(trn),dtype=float)
Ft=np.column_stack([fn for k in range(1,K+1) for fn in [np.sin(2*np.pi*k*tt/12),np.cos(2*np.pi*k*tt/12)]])
m2,s2=trn.mean(),trn.std(); yn2=(trn-m2)/s2
with pm.Model():
    sl=pm.HalfNormal('sl',0.3); sz=pm.HalfNormal('sz',0.15); so=pm.HalfNormal('so',0.3); nu=pm.Gamma('nu',2,0.2)
    l0=pm.Normal('l0',float(yn2[0]),1); li=pm.Normal('li',0,sl,shape=len(trn)-1)
    lev=pm.Deterministic('lev',pt.concatenate([[l0],l0+pt.cumsum(li)]))
    b0=pm.Normal('b0',0,0.5,shape=2*K); bi=pm.Normal('bi',0,sz,shape=(len(trn)-1,2*K))
    bw=pm.Deterministic('bw',pt.concatenate([b0[None,:],b0[None,:]+pt.cumsum(bi,axis=0)],axis=0))
    pm.StudentT('obs',nu=nu,mu=lev+pt.sum(bw*pt.as_tensor_variable(Ft),axis=1),sigma=so,observed=yn2)
    ido=pm.sample(500,tune=500,chains=2,cores=1,target_accept=0.9,progressbar=False,random_seed=SEED)
ll=ido.posterior['lev'].values.reshape(-1,len(trn))[:,-1]
bl=ido.posterior['bw'].values.reshape(-1,len(trn),2*K)[:,-1,:]
tf=np.arange(len(trn),len(trn)+H,dtype=float)
Ff=np.column_stack([fn for k in range(1,K+1) for fn in [np.sin(2*np.pi*k*tf/12),np.cos(2*np.pi*k*tf/12)]])
fb=(ll[:,None]+bl@Ff.T).mean(0)*s2+m2; mae_b_oos=np.mean(np.abs(te-fb))

fig,ax=plt.subplots(figsize=(10,4))
ax.plot(datas[:-H],trn,color=AZUL,lw=1.6,label='Treino')
ax.plot(datas[-H:],te,'o-',color='black',lw=1.8,ms=5,label='Teste real')
ax.plot(datas[-H:],fc,'s--',color=VERM,lw=1.6,ms=4,label=f'SARIMA (MAE={mae_oos:.0f})')
ax.plot(datas[-H:],fb,'^--',color=VERDE,lw=1.6,ms=4,label=f'BTVC (MAE={mae_b_oos:.0f})')
ax.axvline(datas.iloc[-H],color='gray',ls=':',lw=1); ax.yaxis.set_major_formatter(fmt); ax.legend()
ax.set_title('Validação out-of-sample (últimos 6 meses)',fontweight='bold'); plt.show()
print(f"SARIMA OOS MAE=R${mae_oos:.0f} | BTVC OOS MAE=R${mae_b_oos:.0f}")